<a href="https://colab.research.google.com/github/TuanSoCode/SteamData-Pipeline/blob/main/tool_luoibieng.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%writefile /content/drive/MyDrive/csv_helper.py
import pandas as pd
import csv
import re

def ba_khi(filepath):
    with open(filepath, encoding="utf-8") as f:
        r = csv.reader(f)
        h = next(r)
        row = next(r)
    if len(h) != len(row):
        nghi_van = [(i, c) for i, c in enumerate(h) if re.search(r'([a-z])([A-Z])', c)]
        print("Các cột nghi vấn:", nghi_van)
    return pd.read_csv(filepath, sep=",", engine="python", on_bad_lines="error")

def ba_dao(filepath, ten_cu, ten_moi_list):
    df_tam = pd.read_csv(filepath, sep=",", engine="python", on_bad_lines="error")
    cols = df_tam.columns.tolist()
    i = cols.index(ten_cu)
    cols[i:i+1] = ten_moi_list
    return pd.read_csv(filepath, header=0, names=cols,
                        sep=",", engine="python", on_bad_lines="error")

def export_files(df, base_filename):
    try:
        df.to_csv(f"{base_filename}.csv", index=False, encoding='utf-8-sig')
        df.to_parquet(f"{base_filename}.parquet", index=False)
        df.to_json(f"{base_filename}.json", orient='records', force_ascii=False, indent=4)

        print(f"✅ Đã xuất bộ 3 file: {base_filename} (CSV, Parquet, JSON)")
    except Exception as e:
        print(f"⚠️ Lỗi xuất file thô: {e}")

def export_styled_excel(df, filename):
    try:
        df_excel = df.copy()

        def clean_text(x):
            if isinstance(x, str):
                x = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', x)
                return x[:32000]
            return x

        for col in df_excel.select_dtypes(include=['object']).columns:
            df_excel[col] = df_excel[col].apply(clean_text)

        with pd.ExcelWriter(f"{filename}.xlsx", engine='xlsxwriter') as writer:
            df_excel.to_excel(writer, sheet_name='Data', index=False)
            worksheet = writer.sheets['Data']
            workbook = writer.book

            header_format = workbook.add_format({
                'bold': True,
                'fg_color': '#4F81BD',
                'font_color': 'white',
                'border': 1,
                'align': 'center'
            })

            for col_num, value in enumerate(df_excel.columns.values):
                worksheet.write(0, col_num, value, header_format)

            for i, col in enumerate(df_excel.columns):
                column_len = max(df_excel[col].astype(str).map(len).max(), len(col)) + 2
                worksheet.set_column(i, i, min(column_len, 50))

        print(f"✨ Successfully exported styled file: {filename}.xlsx")

    except Exception as e:
        print(f"⚠️ Error exporting Excel: {e}")

Overwriting /content/drive/MyDrive/csv_helper.py
